# Notebook 05: Data Cleaning Pipeline

**Time:** 30 minutes  
**Prerequisites:** Notebook 04 — data collected  
**Goal:** Transform raw collected data into a clean, deduplicated, PII-free training corpus

> 💡 **Why cleaning matters:** "Garbage in, garbage out" is especially true for LLM training.
> LLaMA 4 used MinHash to remove near-duplicates from 40 trillion tokens — even at that scale,
> data quality is the limiting factor for model quality.

## The Cleaning Pipeline

```
Raw text
  → [1] HTML strip & whitespace normalize
  → [2] Language filter (keep only target language)
  → [3] MinHash near-duplicate removal
  → [4] PII detection & anonymisation
  → Clean corpus ✅
```

In [1]:
import os, sys, json
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

from src.data_utils import (
    detect_languages, deduplicate_minhash,
    remove_pii, run_cleaning_pipeline
)
from src.utils import append_to_reflection

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("✅ Setup complete — ready for Notebook 05")

# Load collected data from Notebook 04
arxiv_path = os.path.join(outputs_dir, 'arxiv_clean.json')
if os.path.exists(arxiv_path):
    with open(arxiv_path) as f:
        papers = json.load(f)
    raw_texts = [p.get('raw_text', p.get('abstract', '')) for p in papers]
    raw_texts = [t for t in raw_texts if t.strip()]
    print(f"✅ Loaded {len(raw_texts)} documents from arxiv_clean.json")
else:
    # Fallback: use synthetic data if nb04 wasn't run
    print("⚠️  arxiv_clean.json not found — using synthetic demo data")
    print("   Complete Notebook 04 TODO 1 to use your own data.")
    raw_texts = [
        "Large language models have demonstrated remarkable capabilities in natural language understanding and generation.",
        "Large language models have demonstrated remarkable capabilities in natural language understanding and generation.",  # exact duplicate
        "The transformer architecture, introduced in 2017, uses self-attention mechanisms to capture long-range dependencies.",
        "Transformers use self-attention mechanisms to capture long range dependencies, revolutionizing NLP since 2017.",   # near-duplicate
        "Contact us at john.doe@example.com or call +1-555-123-4567 for support. Your CC: 4111 1111 1111 1111.",
        "Supervised fine-tuning adapts pretrained models to specific tasks using labeled instruction-response pairs.",
        "こんにちは。機械学習は素晴らしい技術です。",   # Japanese (non-English)
        "Alignment techniques like RLHF and DPO ensure model outputs match human preferences and values.",
        "Data deduplication is critical for LLM pretraining to prevent overfitting on repeated content.",
        "  \n\n   ",   # empty / whitespace
    ] * 2   # double to show dedup working

✅ Setup complete — ready for Notebook 05
✅ Loaded 10 documents from arxiv_clean.json


/Users/scottlai/Documents/inferenceAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


---

## Part 1: Language Detection + HTML Cleaning

In [2]:
print("=" * 65)
print("🧪 Experiment 1: Language Detection")
print("=" * 65)
print(f"Detecting languages in {len(raw_texts)} documents...")
print()

lang_distribution = detect_languages(raw_texts)

print()
print("💡 For LLM training data, you typically want to:")
print("   - Keep only languages your model should support")
print("   - Filter out low-resource languages (unless building multilingual model)")
print("   - Maintain proportional representation across languages")

🧪 Experiment 1: Language Detection
Detecting languages in 10 documents...

🌍 LANGUAGE DISTRIBUTION
  en                 10 (100.0%)  ████████████████████
  TOTAL              10

💡 For LLM training data, you typically want to:
   - Keep only languages your model should support
   - Filter out low-resource languages (unless building multilingual model)
   - Maintain proportional representation across languages


In [3]:
print("=" * 65)
print("🧪 Experiment 2: HTML Stripping & Whitespace Normalisation")
print("=" * 65)
print()

import html, re

# Example HTML-contaminated text (common in web-scraped data)
dirty_text = """
<div class="abstract">
  <p>This paper presents a &lt;strong&gt;novel&lt;/strong&gt; approach to 
  &amp;nbsp; machine learning.  \n\n\n
  See: <a href='https://example.com'>link</a>   for   more   info.</p>
</div>
"""

# Cleaning steps
step1 = html.unescape(dirty_text)                          # &lt; → <
step2 = re.sub(r'<[^>]+>', ' ', step1)                    # remove HTML tags
step3 = re.sub(r'\s+', ' ', step2).strip()                # collapse whitespace

print(f"BEFORE (dirty):")
print(f"  {repr(dirty_text[:100])}...")
print()
print(f"AFTER (clean):")
print(f"  {repr(step3)}")
print()
print(f"Length reduction: {len(dirty_text)} → {len(step3)} chars")

🧪 Experiment 2: HTML Stripping & Whitespace Normalisation

BEFORE (dirty):
  '\n<div class="abstract">\n  <p>This paper presents a &lt;strong&gt;novel&lt;/strong&gt; approach to \n '...

AFTER (clean):
  'This paper presents a novel approach to &nbsp; machine learning. See: link for more info.'

Length reduction: 213 → 89 chars


### 🎯 TODO 1: Language Filter Your Corpus

In [4]:
# TODO 1: Decide which language(s) to keep, apply the filter

TARGET_LANGUAGE = "en"   # ← Change if you want to keep a different language

from langdetect import detect, LangDetectException

def filter_by_language(texts, target_lang='en'):
    kept, removed = [], []
    for text in texts:
        text = text.strip()
        if len(text) < 20:
            removed.append(text)
            continue
        try:
            lang = detect(text)
            if lang == target_lang:
                kept.append(text)
            else:
                removed.append(text)
        except LangDetectException:
            removed.append(text)
    return kept, removed

print("=" * 65)
print(f"🎯 TODO 1: Filtering for language='{TARGET_LANGUAGE}'")
print("=" * 65)

kept_texts, removed_texts = filter_by_language(raw_texts, TARGET_LANGUAGE)

print(f"\nResults:")
print(f"  Input:   {len(raw_texts)} documents")
print(f"  Kept:    {len(kept_texts)} documents")
print(f"  Removed: {len(removed_texts)} documents")

todo1_reflection = """
[YOUR REFLECTION HERE]

- What language distribution did you observe in your data?
- Did filtering change anything significant for arXiv data? Why or why not?
- When would you want to keep non-English data?
"""
print()
print(todo1_reflection)

🎯 TODO 1: Filtering for language='en'

Results:
  Input:   10 documents
  Kept:    10 documents
  Removed: 0 documents


[YOUR REFLECTION HERE]

- What language distribution did you observe in your data?
- Did filtering change anything significant for arXiv data? Why or why not?
- When would you want to keep non-English data?



---

## Part 2: MinHash Near-Duplicate Detection

**Why deduplication matters:** Repeated training data causes the model to overfit — it memorises text instead of learning general patterns. Meta found that deduplication improved LLaMA's quality more than adding more raw data.

**MinHash** works by:
1. Converting each document to a set of character n-grams (shingles)
2. Creating a compact signature (the "MinHash") that estimates Jaccard similarity
3. Using LSH (Locality Sensitive Hashing) to find near-duplicates in O(n) time

In [5]:
print("=" * 65)
print("🧪 Experiment 3: MinHash Near-Duplicate Detection")
print("=" * 65)
print()

# Create a synthetic set with known duplicates to verify the algorithm
synthetic_docs = [
    "Large language models are trained on massive corpora using next-token prediction objectives.",
    "Large language models are trained on massive corpora using next-token prediction objectives.",          # exact duplicate
    "Large language models are trained on enormous text corpora using next token prediction objectives.",   # near-duplicate (few words changed)
    "Transformer architectures use self-attention mechanisms to capture contextual representations.",
    "Self-attention in transformers allows models to capture contextual representations efficiently.",       # near-duplicate
    "Reinforcement learning from human feedback aligns language models with user preferences.",
    "Data quality is more important than data quantity for training high-performance language models.",
    "The attention mechanism computes weighted sums of value vectors based on query-key similarity.",
]

print(f"Input: {len(synthetic_docs)} synthetic documents")
print("(includes 2 exact duplicates and 2 near-duplicates)")
print()

deduped_docs, removed_idx = deduplicate_minhash(
    texts=synthetic_docs,
    threshold=0.7,
    num_perm=128
)

print()
print("Removed document indices:", removed_idx)
for i in removed_idx:
    print(f"  [{i}] '{synthetic_docs[i][:80]}...'")

🧪 Experiment 3: MinHash Near-Duplicate Detection

Input: 8 synthetic documents
(includes 2 exact duplicates and 2 near-duplicates)

🔁 MINHASH DEDUPLICATION RESULTS
  Input documents:    8
  Duplicates removed: 2
  Output documents:   6
  Removal rate:       25.0%
  Threshold:          Jaccard ≥ 0.7

Removed document indices: [1, 2]
  [1] 'Large language models are trained on massive corpora using next-token prediction...'
  [2] 'Large language models are trained on enormous text corpora using next token pred...'


### 🎯 TODO 2: Deduplicate Your Real Corpus

In [6]:
# TODO 2: Run deduplication on your real collected corpus

print("=" * 65)
print("🎯 TODO 2: Deduplicating My Corpus")
print("=" * 65)
print()

texts_to_dedup = kept_texts if kept_texts else raw_texts
print(f"Running MinHash on {len(texts_to_dedup)} documents...")
print()

if len(texts_to_dedup) > 1:
    deduped_real, removed_real = deduplicate_minhash(
        texts=texts_to_dedup,
        threshold=0.7
    )
else:
    deduped_real = texts_to_dedup
    removed_real = []

todo2_reflection = """
[YOUR REFLECTION HERE]

- What percentage of your corpus was deduplicated?
- Did you expect more or fewer duplicates? Why?
- How would you adjust the threshold (0.7) to be more strict vs. more lenient?
- arXiv abstracts are often cross-posted — did you find any from the same paper?
"""
print()
print(todo2_reflection)

🎯 TODO 2: Deduplicating My Corpus

Running MinHash on 10 documents...

🔁 MINHASH DEDUPLICATION RESULTS
  Input documents:    10
  Duplicates removed: 0
  Output documents:   10
  Removal rate:       0.0%
  Threshold:          Jaccard ≥ 0.7


[YOUR REFLECTION HERE]

- What percentage of your corpus was deduplicated?
- Did you expect more or fewer duplicates? Why?
- How would you adjust the threshold (0.7) to be more strict vs. more lenient?
- arXiv abstracts are often cross-posted — did you find any from the same paper?



---

## Part 3: PII Detection and Removal

**Why PII matters for LLM training:**
- LLMs can memorise and reproduce training data verbatim
- A model trained on emails containing SSNs could output real SSNs
- GDPR and data protection laws require removal of personal information

**Presidio** (Microsoft) is the industry standard for PII detection:

In [7]:
print("=" * 65)
print("🧪 Experiment 4: PII Detection with Presidio")
print("=" * 65)
print()

# Synthetic paragraph with intentional PII for demonstration
pii_text = """
Dr. Sarah Johnson from Stanford University submitted the paper on January 15, 2024.
For inquiries, contact sarah.johnson@stanford.edu or call +1-650-723-4800.
Her ORCID is 0000-0002-1234-5678. The research was funded by grant NSF-1234567.
Do NOT share: SSN 123-45-6789, Credit Card 4532-0151-1283-0366.
Server IP: 192.168.1.100. Last login: 2024-01-15T10:30:00Z.
"""

print("BEFORE (with PII):")
print(pii_text)
print()

anonymized_text, detected_entities = remove_pii(pii_text)

print()
print("AFTER (PII removed):")
print(anonymized_text)
print()
print(f"Detected {len(detected_entities)} PII entities:")
for ent in detected_entities:
    print(f"  [{ent['entity_type']:<20}] '{ent['text_snippet']}'  (score: {ent['score']})")

🧪 Experiment 4: PII Detection with Presidio

BEFORE (with PII):

Dr. Sarah Johnson from Stanford University submitted the paper on January 15, 2024.
For inquiries, contact sarah.johnson@stanford.edu or call +1-650-723-4800.
Her ORCID is 0000-0002-1234-5678. The research was funded by grant NSF-1234567.
Do NOT share: SSN 123-45-6789, Credit Card 4532-0151-1283-0366.
Server IP: 192.168.1.100. Last login: 2024-01-15T10:30:00Z.



AFTER (PII removed):

Dr. <PERSON> from Stanford University submitted the paper on January 15, 2024.
For inquiries, contact <EMAIL_ADDRESS> or call <PHONE_NUMBER>.
Her ORCID is 0000-0002-1234-5678. The research was funded by grant NSF-1234567.
Do NOT share: SSN 123-45-6789, Credit Card <CREDIT_CARD>.
Server IP: <IP_ADDRESS>. Last login: 2024-01-15T10:30:00Z.


Detected 6 PII entities:
  [EMAIL_ADDRESS       ] 'sarah.johnson@stanford.edu'  (score: 1.0)
  [CREDIT_CARD         ] '4532-0151-1283-0366'  (score: 1.0)
  [IP_ADDRESS          ] '192.168.1.100'  (score: 0.

### 🎯 TODO 3: PII Scan Your Real Corpus

In [8]:
# TODO 3: Run PII detection on your real corpus

print("=" * 65)
print("🎯 TODO 3: PII Scan of My Corpus")
print("=" * 65)
print()

texts_for_pii = deduped_real[:20]  # scan first 20 to keep it fast
print(f"Scanning {len(texts_for_pii)} documents for PII...")
print()

total_entities = []
for text in texts_for_pii:
    _, entities = remove_pii(text)
    total_entities.extend(entities)

if total_entities:
    from collections import Counter
    entity_types = Counter(e['entity_type'] for e in total_entities)
    print(f"Found {len(total_entities)} PII entities:")
    for etype, count in entity_types.most_common():
        print(f"  {etype:<25} {count}")
else:
    print("✅ No PII detected in sampled documents")
    print("   (arXiv abstracts rarely contain PII — as expected)")

todo3_reflection = """
[YOUR REFLECTION HERE]

- What types of PII were found (if any) in your corpus?
- arXiv abstracts rarely have PII — what data sources WOULD have PII?
  (Think: web forums, social media, customer reviews, medical records)
- What are the trade-offs of aggressive PII removal?
  (e.g., removing 'Einstein at Princeton' removes useful factual information)
"""
print()
print(todo3_reflection)

🎯 TODO 3: PII Scan of My Corpus

Scanning 10 documents for PII...

Found 27 PII entities:
  PERSON                    25
  LOCATION                  2


[YOUR REFLECTION HERE]

- What types of PII were found (if any) in your corpus?
- arXiv abstracts rarely have PII — what data sources WOULD have PII?
  (Think: web forums, social media, customer reviews, medical records)
- What are the trade-offs of aggressive PII removal?
  (e.g., removing 'Einstein at Princeton' removes useful factual information)



---

## Part 4: End-to-End Cleaning Pipeline

In [9]:
print("=" * 65)
print("🧪 Experiment 5: Full Cleaning Pipeline")
print("=" * 65)
print()

pipeline_result = run_cleaning_pipeline(
    raw_texts=raw_texts,
    lang_filter="en",
    dedup_threshold=0.7,
    output_dir=os.path.join('..', 'outputs')
)

clean_texts = pipeline_result['clean_texts']
stats       = pipeline_result['stats']

print()
print(f"Final corpus: {len(clean_texts)} documents, ~{stats.get('est_tokens', 0):,} tokens")

🧪 Experiment 5: Full Cleaning Pipeline


📊 Cleaning pipeline — 10 input documents

[Stage 1] HTML stripping & whitespace normalisation...
  → 10 docs remaining

[Stage 2] Language filtering (keeping: 'en')...
  → 10 docs remaining

[Stage 3] MinHash deduplication (threshold=0.7)...
🔁 MINHASH DEDUPLICATION RESULTS
  Input documents:    10
  Duplicates removed: 0
  Output documents:   10
  Removal rate:       0.0%
  Threshold:          Jaccard ≥ 0.7
  → 10 docs remaining

[Stage 4] PII removal...
  → 32 PII entities anonymised

✓ Clean corpus saved: ../outputs/clean_corpus.txt
✓ Stats saved:        ../outputs/corpus_stats.md

✅ PIPELINE COMPLETE
  0_original                         10 docs
  1_after_html_strip                 10 docs
  2_after_lang_filter                10 docs
  3_after_dedup                      10 docs
  4_after_pii                        10 docs
  Estimated tokens                4,975

Final corpus: 10 documents, ~4,975 tokens


### 🎯 TODO 4: Pipeline Reflection

In [10]:
# TODO 4: Reflect on each decision in the pipeline

todo4_reflection = """
[YOUR REFLECTION HERE — answer each question]

1. Language filtering:
   - What threshold did you use? Why?
   - Would you change anything for a multilingual project?

2. Deduplication threshold (0.7 Jaccard):
   - Does 0.7 seem right for your data? What would 0.9 or 0.5 give you?
   - Near-duplication in academic papers: citations, related work, rewrites — OK to dedup?

3. PII removal:
   - Would your production system need more aggressive PII removal? Less?
   - What entity types are most important to anonymize for your project domain?

4. Overall pipeline:
   - What percentage of original data survived all cleaning stages?
   - Does that ratio make sense given the data source?
   - What additional cleaning steps would you add for production use?
"""

print("=" * 65)
print("🎯 TODO 4: Pipeline Decisions & Reflections")
print("=" * 65)
print()
print(todo4_reflection)

# Save full reflection
full_reflection = f"""
### Pipeline Statistics

| Stage | Documents |
|---|---|
"""
for stage, count in stats.get('stage_counts', {}).items():
    full_reflection += f"| {stage} | {count} |\n"

full_reflection += f"""

### TODO 1 — Language Filtering
{todo1_reflection.strip()}

### TODO 2 — MinHash Deduplication
{todo2_reflection.strip()}

### TODO 3 — PII Scan
{todo3_reflection.strip()}

### TODO 4 — Pipeline Reflection
{todo4_reflection.strip()}
"""

reflection_file = append_to_reflection(
    notebook="05",
    section_title="Data Cleaning Pipeline Decisions",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)
print(f"\n✅ Reflection saved: {reflection_file}")

🎯 TODO 4: Pipeline Decisions & Reflections


[YOUR REFLECTION HERE — answer each question]

1. Language filtering:
   - What threshold did you use? Why?
   - Would you change anything for a multilingual project?

2. Deduplication threshold (0.7 Jaccard):
   - Does 0.7 seem right for your data? What would 0.9 or 0.5 give you?
   - Near-duplication in academic papers: citations, related work, rewrites — OK to dedup?

3. PII removal:
   - Would your production system need more aggressive PII removal? Less?
   - What entity types are most important to anonymize for your project domain?

4. Overall pipeline:
   - What percentage of original data survived all cleaning stages?
   - Does that ratio make sense given the data source?
   - What additional cleaning steps would you add for production use?


✅ Reflection saved: ../outputs/homework_reflection.md


## ✅ Notebook 05 Complete!

**What you accomplished:**
- ✅ Detected language distribution in your corpus
- ✅ Stripped HTML and normalised whitespace
- ✅ Ran MinHash near-duplicate detection
- ✅ Scanned for and removed PII using Presidio
- ✅ Ran the full end-to-end cleaning pipeline
- ✅ Saved `outputs/clean_corpus.txt` and `outputs/corpus_stats.md`

**Key concepts:**
- Data cleaning = multiple sequential stages, each reducing noise differently
- MinHash LSH enables near-duplicate detection at O(n) scale
- PII removal is both an ethical and legal requirement for production LLM training
- Even with aggressive cleaning, it's common to retain only 60–80% of scraped data

**Next:** Open **Notebook 06: Fine-tuning & Alignment Concepts** ⚙️